# Nested Booking Limits — Interactive Dashboard
Run either collapsed dashboard cell using its play button; Python code is hidden by default.


In [ ]:
#@title Nested Booking Limits — Interactive Dashboard { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

FARES = [('Promo',79), ('Saver',120), ('Standard',190), ('Flex',320)]
CAPACITY = 180
INITIAL_SOLD = [20,20,20,20]
INITIAL_LIMITS = [60,100,140,180]
# Identical illustrative future requests for both policies: low fares arrive first.
REQUESTS = [55,35,30,40]
sold = INITIAL_SOLD.copy()

css = widgets.HTML("""<style>
.rm{max-width:1050px;font-family:Arial,sans-serif;color:#14364a}
.rm h2{margin:5px 0}.rm h3{margin:16px 0 8px}
.rm small,.rm-note{color:#64748b;font-size:12px}
.rm-stats{display:flex;gap:10px;flex-wrap:wrap;margin:14px 0}
.rm-stat{flex:1;min-width:140px;background:#eef3f8;border-radius:10px;padding:13px}
.rm-stat b{display:block;font-size:25px;margin:3px 0}
.rm table{border-collapse:collapse;width:100%;font-size:14px;margin:12px 0}
.rm th{background:#e8f1f7;text-align:left;padding:9px}
.rm td{padding:9px;border-bottom:1px solid #dbe2e8}
.rm-good{color:#087e4e;font-weight:bold}.rm-bad{color:#b91c1c;font-weight:bold}
.rm-track{height:12px;background:#e5e7eb;border-radius:8px;overflow:hidden}
.rm-fill{height:100%;background:#08739a}
</style>""")
output = widgets.Output()
controls = widgets.Output()
buttons = []
sliders = [
    widgets.IntSlider(value=v,min=0,max=180,step=5,description=label,
        style={'description_width':'125px'},layout=widgets.Layout(width='430px'),
        continuous_update=False)
    for v,label in [(60,'Promo limit'),(100,'Saver limit'),(140,'Standard limit')]
]

def limits():
    return [w.value for w in sliders]+[CAPACITY]

def revenue(counts):
    return sum(n*fare for n,(_,fare) in zip(counts,FARES))

def available(counts, caps):
    left=CAPACITY-sum(counts)
    return [max(0,min(left,caps[i]-sum(counts[:i+1]))) for i in range(4)]

def forecast(nested):
    """Apply protection to future requests, never undo existing sales."""
    counts=sold.copy()
    accepted=[0]*4
    rejected=[0]*4
    caps=limits() if nested else [CAPACITY]*4
    for i,n in enumerate(REQUESTS):
        for _ in range(n):
            if sum(counts)<CAPACITY and (not nested or sum(counts[:i+1])<caps[i]):
                counts[i]+=1
                accepted[i]+=1
            else:
                rejected[i]+=1
    return counts,accepted,rejected

def render():
    caps=limits()
    avail=available(sold,caps)
    total=sum(sold)
    rows=''.join(
        f'<tr><td><b>{name}</b><br><small>USD {fare}</small></td>'
        f'<td>{sold[i]}</td><td>{caps[i]}</td><td>{avail[i]}</td>'
        f'<td>USD {sold[i]*fare:,}</td>'
        f'<td class="{"rm-good" if avail[i] else "rm-bad"}">'
        f'{"OPEN" if avail[i] else "CLOSED"}</td></tr>'
        for i,(name,fare) in enumerate(FARES)
    )
    protected,pa,pr=forecast(True)
    baseline,ba,br=forecast(False)
    delta=revenue(protected)-revenue(baseline)
    compare_rows=''.join(
        f'<tr><td>{name}</td><td>{pa[i]}</td><td>{ba[i]}</td>'
        f'<td>{pr[i]}</td><td>{br[i]}</td></tr>'
        for i,(name,_) in enumerate(FARES)
    )
    sign='+' if delta>0 else '-' if delta<0 else ''
    html=f"""
    <div class="rm">
      <small>ILLUSTRATIVE REVENUE MANAGEMENT DASHBOARD</small>
      <h2>Nested Booking Limits | Flight Inventory Control</h2>
      <small>180 seats · Economy cabin · D−14 (14 days before departure) · Illustrative fares and demand</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>Seats sold</small><b>{total}</b></div>
        <div class="rm-stat"><small>Remaining seats</small><b>{CAPACITY-total}</b></div>
        <div class="rm-stat"><small>Load factor</small><b>{total/CAPACITY:.1%}</b></div>
        <div class="rm-stat"><small>Ticket revenue to date</small><b>USD {revenue(sold):,}</b></div>
      </div>
      <div class="rm-track"><div class="rm-fill" style="width:{100*total/CAPACITY:.2f}%"></div></div>
      <table><thead><tr><th>Fare class</th><th>Sold</th><th>Nested limit</th>
      <th>Available</th><th>Revenue to date</th><th>Status</th></tr></thead>
      <tbody>{rows}</tbody></table>
      <div class="rm-note">Available counts overlap and cannot be added. Ticket revenue excludes ancillary sales and costs.</div>
      <h3>Revenue comparison: identical future booking requests</h3>
      <small>From D−14 through departure, future requests arrive in order: Promo {REQUESTS[0]},
        Saver {REQUESTS[1]}, Standard {REQUESTS[2]}, Flex {REQUESTS[3]}.
        Only the inventory policy changes.</small>
      <div class="rm-stats">
        <div class="rm-stat"><small>WITH nested protection · final ticket revenue</small>
          <b>USD {revenue(protected):,}</b><small>{sum(protected)} total seats sold</small></div>
        <div class="rm-stat"><small>WITHOUT protection · first come, first served</small>
          <b>USD {revenue(baseline):,}</b><small>{sum(baseline)} total seats sold</small></div>
      </div>
      <div class="{"rm-good" if delta>=0 else "rm-bad"}">
      Revenue difference (protection minus no protection): {sign}USD {abs(delta):,}</div>
      <table><thead><tr><th>Future fare requests</th>
        <th>Accepted with protection</th><th>Accepted without</th>
        <th>Rejected with protection</th><th>Rejected without</th>
        </tr></thead><tbody>{compare_rows}</tbody></table>
      <div class="rm-note">Deterministic classroom scenario, not an optimized or
      statistically expected-revenue forecast. Protection can help or hurt when
      demand changes. Protection applies to future bookings; past sales remain.</div>
    </div>"""
    with output:
        clear_output(wait=True)
        display(HTML(html))
    for i,button in enumerate(buttons):
        button.disabled=avail[i]<=0

def slider_changed(change=None):
    # Cumulative limits must remain nondecreasing.
    a,b,c=sliders
    a.max=b.value
    b.min=a.value
    b.max=c.value
    c.min=b.value
    render()

for slider in sliders:
    slider.observe(slider_changed,names='value')

def book(i):
    if available(sold,limits())[i]>0:
        sold[i]+=1
        render()

for i,(name,fare) in enumerate(FARES):
    button=widgets.Button(description=f'+1 {name} (USD {fare})',
                          layout=widgets.Layout(width='205px'))
    button.on_click(lambda _,i=i:book(i))
    buttons.append(button)

reset=widgets.Button(description='↻ Reset example',button_style='info',
                     layout=widgets.Layout(width='185px'))
def reset_all(_):
    sold[:]=INITIAL_SOLD
    sliders[0].value=0
    sliders[1].value=0
    sliders[2].value=INITIAL_LIMITS[2]
    sliders[1].value=INITIAL_LIMITS[1]
    sliders[0].value=INITIAL_LIMITS[0]
    render()
reset.on_click(reset_all)

display(css,output,
    widgets.HTML('<b>Accept one booking:</b>'),
    widgets.HBox(buttons,layout=widgets.Layout(flex_flow='row wrap',gap='8px')),
    widgets.HTML('<b>Adjust nested booking limits (cumulative):</b>'),
    widgets.VBox(sliders),reset)
slider_changed()


## Activity 2 — Seat Protection: Three Stages

**Scenario:** A 180-seat flight has **160 seats booked** and **20 available**. Fifteen Saver customers request tickets today at **$120**. Later Flex customers may request tickets at **$320**. Ignore revenue from the 160 passengers already booked.

### Stage 1 — Four fixed decisions

| Later Flex requests | 0 | 5 | 10 | 15 | 20 |
|---|---:|---:|---:|---:|---:|
| Probability | 15% | 20% | 30% | 25% | 10% |

Compare **A:** accept 15 Saver/protect 5, **B:** accept 5/protect 15, **C:** accept none/protect 20, and **D:** accept 10/protect 10.

**Question:** Which decision produces the highest expected incremental revenue? Use the fixed table and immediately visible graph.

### Stage 2 — More demand does not always change the decision

Click **Continue to Stage 2**. Compare the three forecast distributions below using the sliders and the revenue graph.

| Chance of any Flex demand | 0 requests | 5 | 10 | 15 | 20 |
|---|---:|---:|---:|---:|---:|
| 65% | 35% | 13.2% | 16.5% | 22.3% | 13% |
| 85% | 15% | 20% | 30% | 25% | 10% |
| 95% | 5% | 51.2% | 24.2% | 16.7% | 2.8% |

**Questions:** How does expected revenue change? Does the optimal number of seats protected change? Why can revenue fall even when the probability of some Flex demand increases?

### Stage 3 — Demand size matters, not only probability

Click **Continue to Stage 3**. This stage uses **different, illustrative distributions**: at 65% the demand that occurs is generally larger, while at 95% it is generally smaller. The notebook shows the updated probabilities for 0, 5, 10, 15 and 20 requests.

**Questions:** Compare the preferred protection levels at 65%, 85%, and 95%. Why can a lower chance of demand justify protecting *more* seats? What does this reveal about demand forecasting?

**Interpretation of the slider:** "Chance of any Flex demand" means the chance of **at least one** later Flex request, **not** the chance that all remaining seats sell. The distribution of request quantities also varies across the scenarios; these are two separate assumptions.

*For the 95% Stage 2 row, the displayed percentages total 99.9% due to rounding; calculations normalize the row to 100%.*

In [ ]:
#@title Activity 2 — Three Stages of Seat Protection { display-mode: "form" }
import ipywidgets as widgets
from IPython.display import HTML, display, clear_output
import matplotlib.pyplot as plt

SEATS_LEFT, SAVER_REQUESTS = 20, 15
SAVER_FARE, FLEX_FARE = 120, 320
DEMAND_LEVELS = [0, 5, 10, 15, 20]
STAGE1_PROBS = [0.15, 0.20, 0.30, 0.25, 0.10]

# Stage 2: user-specified probability distributions.
# The 95% row is rounded to 99.9%; normalize the positive part internally.
STAGE2_ANCHORS = [
    (0.65, [0.35, 0.132, 0.165, 0.223, 0.13]),
    (0.85, [0.15, 0.20, 0.30, 0.25, 0.10]),
    (0.95, [0.05, 0.512, 0.242, 0.167, 0.028])
]
# Stage 3: contrasting demand-size patterns, independent of likelihood.
# More probable demand may be smaller, and less probable demand may be larger.
STAGE3_ANCHORS = [
    (0.65, [0.35, 0.65*0.05, 0.65*0.10, 0.65*0.65, 0.65*0.20]),
    (0.85, [0.15, 0.40, 0.10, 0.25, 0.10]),
    (0.95, [0.05, 0.95*0.75, 0.95*0.15, 0.95*0.07, 0.95*0.03])
]

def distribution(stage, chance_fraction):
    anchors = STAGE2_ANCHORS if stage == 2 else STAGE3_ANCHORS
    normalized = []
    for q, raw in anchors:
        # Keep P(0 requests) = 1-q exactly; distribute q over positive levels.
        positive_sum = sum(raw[1:])
        normalized.append((q, [1-q] + [q*p/positive_sum for p in raw[1:]]))
    if chance_fraction <= normalized[0][0]:
        positive = [p/normalized[0][0] for p in normalized[0][1][1:]]
    elif chance_fraction >= normalized[-1][0]:
        positive = [p/normalized[-1][0] for p in normalized[-1][1][1:]]
    else:
        for (p0,a),(p1,b) in zip(normalized[:-1],normalized[1:]):
            if p0 <= chance_fraction <= p1:
                t=(chance_fraction-p0)/(p1-p0)
                positive = [(1-t)*u/p0 + t*v/p1 for u,v in zip(a[1:],b[1:])]
                break
    return [1-chance_fraction] + [chance_fraction*v for v in positive]

def evaluate(protected, probabilities):
    saver_accepted = min(SAVER_REQUESTS, SEATS_LEFT-protected)
    today = saver_accepted*SAVER_FARE
    capacity_for_flex = SEATS_LEFT-saver_accepted
    later = sum(p*min(d,capacity_for_flex)*FLEX_FARE
                for d,p in zip(DEMAND_LEVELS,probabilities))
    return saver_accepted,today,later,today+later

policies=[
    ('A · Accept 15 Saver / protect 5',5),
    ('B · Accept 5 Saver / protect 15',15),
    ('C · Accept 0 Saver / protect 20',20),
    ('D · Accept 10 Saver / protect 10',10)
]
fixed_rows=''.join(
    '<tr><td style="text-align:left;padding:8px;border-bottom:1px solid #ddd">' + label + '</td>' +
    f'<td>{evaluate(n,STAGE1_PROBS)[0]}</td>' +
    f'<td>USD {evaluate(n,STAGE1_PROBS)[1]:,.0f}</td>' +
    f'<td>USD {evaluate(n,STAGE1_PROBS)[2]:,.0f}</td>' +
    f'<td><b>USD {evaluate(n,STAGE1_PROBS)[3]:,.0f}</b></td></tr>'
    for label,n in policies
)
display(HTML("""
<div style="font-family:Arial;max-width:950px;color:#103952">
<h3>Stage 1 · Four decisions under a fixed forecast</h3>
<p>180 seats · 160 already booked · Saver USD 120 · Flex USD 320</p>
<p>Later Flex demand: 0 (15%), 5 (20%), 10 (30%), 15 (25%), 20 (10%).</p>
<table style="width:100%;border-collapse:collapse;text-align:right">
<tr style="background:#e8f1f7"><th style="text-align:left">Decision</th>
<th>Saver accepted</th><th>Revenue today</th>
<th>Expected later Flex revenue</th><th>Expected total</th></tr>""" +
fixed_rows + """
</table><p style="font-size:12px;color:#64748b">Incremental revenue only;
exclude the 160 existing bookings.</p></div>"""))

stage = {'value':1}
seats = widgets.IntSlider(value=15,min=5,max=20,step=1,description='Seats protected',
    style={'description_width':'170px'},layout=widgets.Layout(width='510px'),
    continuous_update=False)
chance = widgets.IntSlider(value=85,min=0,max=100,step=5,
    description='Chance of any Flex demand',
    style={'description_width':'170px'},layout=widgets.Layout(width='510px'),
    continuous_update=False)
output = widgets.Output()
to2 = widgets.Button(description='Continue to Stage 2 →',button_style='info',
    layout=widgets.Layout(width='235px'))
to3 = widgets.Button(description='Continue to Stage 3 →',button_style='info',
    layout=widgets.Layout(width='235px'))
reset = widgets.Button(description='Reset this stage',
    layout=widgets.Layout(width='170px'))

stage_instructions = widgets.HTML()
controls = widgets.VBox([stage_instructions,seats,chance,reset])
controls.layout.display='none'
to3.layout.display='none'

def draw(change=None):
    current_stage=stage['value']
    p=STAGE1_PROBS if current_stage==1 else distribution(current_stage,chance.value/100)
    n=seats.value
    saver,today,later,total=evaluate(n,p)
    xs=list(range(5,21))
    ys=[evaluate(x,p)[3] for x in xs]
    best=max(ys)
    best_n=[x for x,y in zip(xs,ys) if abs(y-best)<1e-6]
    with output:
        clear_output(wait=True)
        if current_stage>=2:
            headers=''.join(f'<th style="padding:8px">{d} requests</th>' for d in DEMAND_LEVELS)
            values=''.join(f'<td style="padding:8px;text-align:center">{v:.1%}</td>' for v in p)
            display(HTML(
                '<div style="font-family:Arial;max-width:940px;color:#103952">'
                f'<p><b>Current forecast:</b> {chance.value}% chance of some Flex demand; '
                f'{100-chance.value}% chance of zero requests.</p>'
                '<table style="width:100%;border-collapse:collapse">'
                f'<tr style="background:#e8f1f7">{headers}</tr><tr>{values}</tr></table>'
                '<p style="font-size:12px;color:#64748b">Probability rows are '
                'normalized for calculations. The positive-demand mix changes across '
                'scenarios, not only the probability of demand occurring.</p></div>'
            ))
        display(HTML(
            '<div style="font-family:Arial;max-width:940px;color:#103952">'
            f'<p><b>Selected decision:</b> protect {n} seats and accept {saver} Saver requests. '
            f'Today USD {today:,.0f} + expected Flex USD {later:,.0f} = '
            f'<b>USD {total:,.0f}</b> expected incremental revenue.</p></div>'
        ))
        fig,ax=plt.subplots(figsize=(9,3.7))
        ax.plot(xs,ys,marker='o',markersize=4,linewidth=2,label='Expected revenue')
        ax.scatter([n],[total],s=100,color='#e77628',zorder=4,label='Selected decision')
        ax.set(xlabel='Seats protected for later Flex requests',
               ylabel='Expected incremental ticket revenue (USD)',
               title=f'Stage {current_stage} · Expected revenue vs seats protected')
        ax.set_xticks(xs)
        ax.grid(alpha=.25)
        ax.legend(loc='lower left',fontsize=9)
        fig.tight_layout()
        display(fig)
        plt.close(fig)

def reveal2(_):
    stage['value']=2
    seats.value=15
    chance.value=85
    stage_instructions.value=(
        '<b>Stage 2 — Probability changes; optimum may stay the same</b>'
        '<p>Compare 65%, 85% and 95%. The demand-size distributions shown in '
        'the table change too. Check whether the optimal number of seats changes.</p>')
    controls.layout.display=''
    to2.layout.display='none'
    to3.layout.display=''
    draw()

def reveal3(_):
    stage['value']=3
    seats.value=15
    chance.value=85
    stage_instructions.value=(
        '<b>Stage 3 — Different demand-size distributions</b>'
        '<p>Again compare 65%, 85% and 95%, but now use a contrasting forecast: '
        'when some Flex demand occurs, it tends to be larger at 65% and smaller '
        'at 95%. Does the best protection level change? '
        'The chance and size of demand are separate forecast assumptions.</p>')
    to3.layout.display='none'
    draw()

def reset_current(_):
    seats.value=15
    chance.value=85
    draw()

seats.observe(draw,names='value')
chance.observe(draw,names='value')
to2.on_click(reveal2)
to3.on_click(reveal3)
reset.on_click(reset_current)
display(output,to2,controls,to3)
draw()
